# Phase 3: Data Preprocessing & Feature Engineering Pipeline
## Leakage Prevention, Column Transformers & 70/15/15 Splitting
---
### Objectives:
1. Enforce strict data partition into **70% Training (28,000)**, **15% Validation (6,000)**, and **15% Testing (6,000)**.
2. Prevent data leakage by fitting all imputation, encoding, and scaling pipelines strictly on training data.
3. Build a Scikit-learn `ColumnTransformer` with `SimpleImputer`, `OneHotEncoder`, and dual scaled/unscaled pathways.
4. Verify transformed feature dimensions and inspect column mappings.



In [1]:
import os
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = os.path.dirname(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data_preprocessing import (
    load_dataset,
    split_data,
    build_preprocessors,
    prepare_preprocessed_datasets,
    CATEGORICAL_FEATURES,
    NUMERICAL_FEATURES,
    TARGET_COLUMN
)
print("Preprocessing modules imported successfully!")



### 1. Dataset Loading and Partitioning (70 / 15 / 15)


In [1]:
df = load_dataset()
print(f"Full Dataset: {df.shape}")

X_train, X_val, X_test, y_train, y_val, y_test = split_data(df, random_state=42)

print(f"Training Set:   X = {X_train.shape}, y = {y_train.shape} (70.0%)")
print(f"Validation Set: X = {X_val.shape},   y = {y_val.shape}   (15.0%)")
print(f"Test Set:       X = {X_test.shape},  y = {y_test.shape}   (15.0%)")



### 2. ColumnTransformer Pipeline Architecture


In [1]:
base_prep, scaled_prep = build_preprocessors()
print("Base Preprocessor (Tree Models):")
print(base_prep)



### 3. Pipeline Fitting on Training Data Only


In [1]:
# Fit strictly on X_train to avoid data leakage
base_prep.fit(X_train)
scaled_prep.fit(X_train)

feature_names = list(base_prep.get_feature_names_out())
print(f"Total Transformed Features: {len(feature_names)}")
for i, name in enumerate(feature_names):
    print(f"  [{i:02d}] {name}")



### 4. Transformation Verification


In [1]:
X_train_trans = base_prep.transform(X_train)
X_val_trans = base_prep.transform(X_val)
X_test_trans = base_prep.transform(X_test)

print(f"Transformed X_train: {X_train_trans.shape}")
print(f"Transformed X_val:   {X_val_trans.shape}")
print(f"Transformed X_test:  {X_test_trans.shape}")
print(f"Any NaN in X_train_trans? {np.isnan(X_train_trans).any()}")



### 5. Summary of Preprocessing Safeguards
- **Zero Leakage:** Means, medians, and one-hot categories learned strictly from the 28,000 training instances.
- **Tree Compatibility:** Unscaled raw features preserved for tree-based algorithms (preserving threshold split semantics).
- **Linear Compatibility:** Dual pipeline `scaled_prep` applies `StandardScaler` after encoding for distance/linear models.

